# MeteoPrep — exploration

Notebook d'exploration de l'étape 2 : météo mensuelle par hôtel.

**Principe :** les champs d'identification (`hotel_code` … `hotel_lat`, `hotel_lon`) viennent de RodPrep.
La météo est calculée au point `(hotel_lat, hotel_lon)` (stations Meteostat les plus proches).

**Années :** si non fournies → **année en cours**. Les mois manquants de l'année en cours
sont complétés par le **même mois de l'année précédente** (jamais d'imputation à 0).


In [8]:
from pathlib import Path
import sys
from datetime import datetime

import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name != "MeteoPrep" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
PREPARE = ROOT.parent
PROJECT = PREPARE.parent

sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(ROOT / "Src"))

INPUT_DIR = ROOT / "Input"
OUTPUT_DIR = ROOT / "Output"
ROD_OUTPUT = PREPARE / "RodPrep" / "Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

print("Année en cours :", datetime.utcnow().year)


Année en cours : 2026


## 1. Entrée — identité hôtel depuis RodPrep

Champs retenus : `hotel_code`, `hotel_name`, `hotel_brand`, `hotel_city`, `hotel_lat`, `hotel_lon`.

On rafraîchit l'entrée depuis RodPrep. Par défaut les années cibles = année en cours
(passer `target_years=(2024, 2025, …)` pour l'historique ventes).


In [ ]:
from meteo_prep.prep import MeteoPrep, HOTEL_IDENTITY_COLS, READABLE_WEATHER, default_target_years

# Années non fournies → année en cours. Pour jointure ventes historiques, ex. :
# prep = MeteoPrep(INPUT_DIR, OUTPUT_DIR, target_years=(2023, 2024, 2025, 2026))
prep = MeteoPrep(INPUT_DIR, OUTPUT_DIR)

if not (ROD_OUTPUT / "hotel_lookup.parquet").exists():
    raise FileNotFoundError("Exécuter d'abord RodPrep/Explore/explore.ipynb")

hotels_path = prep.fill_input_from_rod(ROD_OUTPUT)
print("Entrée créée depuis RodPrep :", hotels_path)
print("Années cibles :", prep.target_years, "(défaut =", default_target_years(), ")")

hotels = prep.load_input()
print(f"Hôtels : {len(hotels)}")
geo_ok = hotels["hotel_lat"].notna() & hotels["hotel_lon"].notna()
print(f"Avec lat/lon : {geo_ok.sum()} / {len(hotels)}")
hotels[[c for c in HOTEL_IDENTITY_COLS if c in hotels.columns]]


## 2. Météo au point hôtel (`hotel_lat`, `hotel_lon`)

`weather_for_hotel` interroge Meteostat au point `(lat, lon)` sur la fenêtre
années cibles + année précédente (pour l'imputation). Résultat indexé par `(annee, mois)`.


In [ ]:
enrich_summary = []
for _, hotel in hotels.iterrows():
    info = prep.weather_for_hotel(hotel)
    by_ym = info.get("weather_by_year_month") or {}
    years = sorted({y for (y, _m) in by_ym.keys()}) if by_ym else []
    enrich_summary.append({
        "hotel_code": info["hotel_code"],
        "hotel_name": info["hotel_name"],
        "hotel_lat": info["hotel_lat"],
        "hotel_lon": info["hotel_lon"],
        "source": info["source"],
        "nb_mois_annee": len(by_ym),
        "annees": ",".join(str(y) for y in years),
        "nb_cles_meteo": info["nb_cles_meteo"],
        "warnings": "; ".join(info["warnings"]) if info["warnings"] else "",
    })

enrich_df = pd.DataFrame(enrich_summary)
print(f"Enrichissements : {len(enrich_df)} hôtels")
enrich_df


## 3. Aperçu — premier hôtel avec coordonnées


In [ ]:
sample = hotels[hotels["hotel_lat"].notna() & hotels["hotel_lon"].notna()]
if sample.empty:
    sample = hotels
sample_hotel = sample.iloc[0]

info = prep.weather_for_hotel(sample_hotel)
by_ym = info.get("weather_by_year_month") or {}

print(
    f"Hôtel {info['hotel_code']} @ ({info['hotel_lat']}, {info['hotel_lon']}) "
    f"— {len(by_ym)} mois×années (source={info['source']})"
)

preview_rows = []
for (year, month), metrics in sorted(by_ym.items()):
    row = {"annee": year, "mois": month}
    for k in sorted(metrics)[:3]:
        row[k] = metrics[k]
    preview_rows.append(row)
pd.DataFrame(preview_rows).head(18)


## 4. Renommage lisible

Métriques déjà en `meteo_{libelle}_{stat}` (mean / min / max). Mapping brut → lisible :


In [ ]:
print("Mapping métriques :", READABLE_WEATHER)
# Compat profil mensuel (année en cours si année absente des clés aplaties)
monthly_readable = prep._readable_monthly(by_ym)
readable_rows = []
for month, metrics in sorted(monthly_readable.items()):
    for col, val in sorted(metrics.items()):
        readable_rows.append({"mois": month, "colonne": col, "valeur": val})
readable_preview = pd.DataFrame(readable_rows)
print(f"Colonnes lisibles (année préférée) : {readable_preview['colonne'].nunique() if not readable_preview.empty else 0}")
readable_preview.head(18)


## 5. Grille `hotel_code × annee × mois`

Une ligne par combinaison pour les années cibles **et** l'année précédente (source d'imputation).


In [ ]:
rows = []
for _, hotel in hotels.iterrows():
    rows.extend(prep._rows_for_hotel(hotel))

expanded = pd.DataFrame(rows)
print(f"Grille brute : {expanded.shape[0]} lignes × {expanded.shape[1]} colonnes")
print(f"Hôtels : {expanded['hotel_code'].nunique()} — années : {sorted(expanded['annee'].unique())}")
expanded.sort_values(["hotel_code", "annee", "mois"]).head(12)


## 6. Imputation — mois manquants ← année précédente

Pour chaque `(hotel_code, annee, mois)` et chaque colonne `meteo_*` :
1. si la valeur est manquante → prendre le **même mois de l'année N-1** (puis N-2, …) ;
2. **jamais** d'imputation à `0`.

Les mois futurs de l'année en cours sont ainsi complétés par l'année dernière.


In [ ]:
meteo_cols = [c for c in expanded.columns if c.startswith("meteo_")]
missing_before = int(expanded[meteo_cols].isna().sum().sum()) if meteo_cols else 0

imputed = prep._impute_missing(expanded)
# Sortie finale : années cibles uniquement
imputed_targets = imputed[imputed["annee"].isin(prep.target_years)].copy()
missing_after = int(imputed_targets[meteo_cols].isna().sum().sum()) if meteo_cols else 0

print(f"NaN meteo avant imputation : {missing_before}")
print(f"NaN meteo après imputation (années cibles) : {missing_after}")
print("(les NaN restants = aucune valeur disponible sur les années antérieures non plus)")
imputed_targets.sort_values(["hotel_code", "annee", "mois"]).head(12)


## 7. Persistance `Output/`

`prep.run()` : entrée → météo lat/lon (année×mois) → imputation N←N-1 → filtre années cibles → fichiers.


In [ ]:
meteo_final = prep.run()
print(f"meteo_monthly : {meteo_final.shape}")
print(f"Années : {sorted(meteo_final['annee'].unique()) if not meteo_final.empty else []}")
print(f"Hôtels : {sorted(meteo_final['hotel_code'].dropna().unique())}")
display_cols = ["hotel_code", "hotel_name", "annee", "mois"] + [
    c for c in meteo_final.columns if c.startswith("meteo_temperature")
]
meteo_final[display_cols].head(12)

print("\nFichiers produits :")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(" ", path.name, f"({path.stat().st_size} octets)")
